In [43]:
import yfinance as yf
import pandas as pd
import numpy as np
import xgboost as xgb

import ta

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [44]:
print("Downloading stock data for ADMR.JK...")
df = yf.download('ADMR.JK', start='2022-01-01')

if isinstance(df.columns, pd. MultiIndex):
    df.columns = df.columns.droplevel(1)
    
tgl_awal = df.index.min()
tgl_akhir = df.index.max() + pd.Timedelta(days=1)

df['ADRO_Close'] = yf.download('ADRO.JK', start=tgl_awal, end=tgl_akhir)['Close']
df['IHSG'] = yf.download('^JKSE', start=tgl_awal, end=tgl_akhir)['Close']
df['USDIDR'] = yf.download('IDR=X', start=tgl_awal, end=tgl_akhir)['Close']
df.head()

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,ADRO_Close,IHSG,USDIDR
Date,,,,,,,,
2022-01-03,128.670883,128.670883,128.670883,128.670883,34289600,840.725952,6665.308105,14215.000000
2022-01-04,173.467422,173.467422,173.467422,173.467422,22170900,815.894470,6695.373047,14283.599609
2022-01-05,232.560715,232.560715,209.685890,209.685890,43203600,794.610107,6662.298828,14385.500000
2022-01-06,289.747772,289.747772,289.747772,289.747772,52097500,815.894470,6653.351074,14446.500000
2022-01-07,362.184723,362.184723,362.184723,362.184723,19248600,862.010071,6701.315918,14408.000000


In [45]:
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_30'] = df['Close'].rolling(window=30).mean()

df['SMA_Diff'] = df['SMA_10'] - df['SMA_30'] # Calculate the difference between the two SMAs for uptrend and downtrend identification

df['RSI'] = ta.momentum.rsi(df['Close'], window=14)

macd_obj = ta.trend.MACD(df['Close'], window_fast=12, window_slow=26, window_sign=9)
df['MACD'] = macd_obj.macd() #blue line
df['MACD_Signal'] = macd_obj.macd_signal() #orange line
df['MACD_Diff'] = macd_obj.macd_diff() #bar chart

#Trailing Support and Resistance
# df['Support'] = df['Low'].rolling(window=14).min().shift(1) 
# df['Resistance'] = df['High'].rolling(window=14).max().shift(1)
# df['Dist_to_Support'] = df['Close'] - df['Support']
# df['Dist_to_Resistance'] = df['Resistance'] - df['Close']

#V-Shape/Support statis 
is_lembah = ( #Requirement: Low Price 2 days ago is lowest  among 2 days ago and 2 days later
    (df['Low'].shift(2) < df['Low'].shift(3)) & # Low than 3 days ago
    (df['Low'].shift(2) < df['Low'].shift(4)) & # Low than 4 days ago
    (df['Low'].shift(2) < df['Low'].shift(1)) & # Low than 1 day ago
    (df['Low'].shift(2) < df['Low'])            # Low than today
)
df['Support_Lembah'] = np.nan
df.loc[is_lembah, 'Support_Lembah'] = df['Low'].shift(2) # Assign the low price from 2 days ago to 'Support_Lembah' if the condition is met

df['Support_Lembah'] = df['Support_Lembah'].ffill()
df['Dist_to_Lembah'] = df['Close'] - df['Support_Lembah']
df.ffill(inplace=True)
df.tail()
# df[['Close', 'Low', 'Support_Lembah', 'Dist_to_Lembah']].tail(15)

Price,Close,High,Low,Open,Volume,ADRO_Close,IHSG,USDIDR,SMA_10,SMA_30,SMA_Diff,RSI,MACD,MACD_Signal,MACD_Diff,Support_Lembah,Dist_to_Lembah
Date,,,,,,,,,,,,,,,,,
2026-03-05,2020.0,2070.0,1990.0,2060.0,34233600,2430.0,7710.537109,16865.000000,2079.0,2017.166667,61.833333,52.059302,52.784742,58.449325,-5.664583,1985.0,35.0
2026-03-06,1980.0,2030.0,1955.0,2000.0,26064100,2400.0,7585.687012,16913.099609,2070.0,2016.500000,53.500000,49.479864,43.637857,55.487031,-11.849174,1925.0,55.0
2026-03-09,1945.0,1950.0,1760.0,1865.0,68683900,2350.0,7337.369141,16923.699219,2055.5,2009.000000,46.500000,47.272723,33.182174,51.026060,-17.843886,1925.0,20.0
2026-03-10,1960.0,1985.0,1905.0,1960.0,46967300,2380.0,7440.913086,16886.000000,2049.5,1998.333333,51.166667,48.336363,25.808836,45.982615,-20.173779,1925.0,35.0
2026-03-11,1990.0,2010.0,1955.0,1975.0,9985100,2420.0,7522.395020,16874.000000,2033.5,1988.666667,44.833333,50.487597,22.131049,41.212302,-19.081253,1760.0,230.0


In [46]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int) # 1 if the next day's close is higher than today's close, else 0
df = df.iloc[:-1] # Remove the last row which has NaN target value
df.dropna(inplace=True) 
# df.shape[0] # Number of rows after dropping NaN values
# df.tail() 

X = df.drop(columns=['Target']) # Features for training (all columns except 'Target')
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print(f"Total clean usable data: {df.shape[0]} days")
print(f"AI training data (Train): {len(X_train)} days")
print(f"AI testing data (Test): {len(X_test)} days")

Total clean usable data: 970 days
AI training data (Train): 776 days
AI testing data (Test): 194 days


In [47]:
#XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=100, # Number of trees in the ensemble
    learning_rate=0.1, # Step size shrinkage used to prevent overfitting
    max_depth=5, # Maximum depth of a tree
    random_state=42, # Seed for reproducibility
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

Model Accuracy: 0.53
